# Tabular claims fraud modeling (synthetic data)

**Not real insurer data.** We use `synthetic_data/claims_fraud_sample.csv` (~800 rows) to practice patterns insurers use: **class imbalance**, **precision/recall**, and **calibration**.

**What you learn:** How fraud detection differs from accuracy; why PR-AUC matters when positives are rare; how `class_weight` and thresholds trade off false positives vs misses.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_recall_curve, auc
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

ROOT = Path('..')
CSV = ROOT / 'synthetic_data' / 'claims_fraud_sample.csv'
df = pd.read_csv(CSV)
df.head()

## 1. EDA — imbalance is the default in fraud

In [ ]:
print('Fraud rate:', df['fraud_label'].mean().round(4))
print(df.describe())
df.groupby('fraud_label').mean(numeric_only=True)

## 2. Features + train/test (stratified)

In [ ]:
feature_cols = ['claim_amount_usd', 'driver_age', 'num_prior_claims', 'policy_tenure_months', 'injury_severity_1to5']
cat_cols = ['region_code']
X = df[feature_cols + cat_cols]
y = df['fraud_label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

preprocess = ColumnTransformer([
    ('num', StandardScaler(), feature_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols),
])
clf = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)),
])
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:, 1]
pred_default = clf.predict(X_test)
print(classification_report(y_test, pred_default, target_names=['clean', 'fraud']))

## 3. PR curve — the right metric when fraud is rare

In [ ]:
precision, recall, thr = precision_recall_curve(y_test, proba)
pr_auc = auc(recall, precision)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=ax[0], name=f'PR AUC={pr_auc:.3f}')
RocCurveDisplay.from_predictions(y_test, proba, ax=ax[1])
plt.tight_layout()
plt.show()
print('PR-AUC:', round(pr_auc, 4))

## 4. What we learned (for interviews)

| Lesson | Why it matters for P&C AI/data roles |
|--------|--------------------------------------|
| **Accuracy lies** | 94% accuracy can mean never predict fraud and still look good. |
| **PR-AUC / recall at k** | Ops cares about catching fraud without drowning in false alarms. |
| **class_weight balanced** | Simple first baseline under imbalance. |
| **Threshold tuning** | Move threshold for target precision or recall. |
| **Not GenAI** | Shows classical ML + LLM breadth — many insurer DS roles want both. |